# Tree-Ring 官方原始环境复现与 governed import 入口

该 Notebook 服务第二条链路: 官方原始环境复现 + governed import。它不把 legacy Stable Diffusion 结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

正式逻辑位于 `paper_workflow/colab_utils/tree_ring_official_reference.py`, Notebook 只负责 Colab 冷启动、参数注入、远程执行和打包。


## 运行边界

1. 默认样本数为 5, 用于与当前 T2SMark 小样本链路测试保持一致。
2. Notebook 会在 helper 中按 `external_baseline/source_registry.json` 自动补齐 Tree-Ring 官方源码快照; 源码快照仍按外部缓存治理, 不进入 git 提交。
3. 若要直接运行官方 Tree-Ring 源码, 建议先准备符合 `external_baseline/primary/tree_ring/source/requirements.txt` 的 legacy Python 环境, 并把 `SLM_WM_TREE_RING_OFFICIAL_PYTHON_EXECUTABLE` 指向该环境的 Python 可执行文件。
4. 若官方原始环境在外部机器完成运行, 可设置 `SLM_WM_TREE_RING_OFFICIAL_RUN_COMMAND=0`, 并通过 `SLM_WM_TREE_RING_OFFICIAL_SUMMARY_IMPORT_PATH` 或 `SLM_WM_TREE_RING_OFFICIAL_LOG_IMPORT_PATH` 导入受治理结果。
5. 该入口默认写入 `/content/drive/MyDrive/SLM/tree_ring_official_reference`, 所有核对文件会纳入压缩包。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/tree_ring_official_reference')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_OUTPUT_DIR', 'outputs/tree_ring_official_reference')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_SOURCE_DIR', 'external_baseline/primary/tree_ring/source')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_RUN_NAME', 'tree_ring_official_legacy_reference')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_SAMPLE_COUNT', '5')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_START_INDEX', '0')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_MODEL_ID', 'stabilityai/stable-diffusion-2-1-base')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_PYTHON_EXECUTABLE', '')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_RUN_COMMAND', '1')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_REQUIRE_CUDA', '1')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_SUMMARY_IMPORT_PATH', '')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_LOG_IMPORT_PATH', '')
os.environ.setdefault('SLM_WM_TREE_RING_OFFICIAL_TIMEOUT_SECONDS', '86400')
print({
    'drive_output_dir': os.environ['SLM_WM_TREE_RING_OFFICIAL_DRIVE_OUTPUT_DIR'],
    'sample_count': os.environ['SLM_WM_TREE_RING_OFFICIAL_SAMPLE_COUNT'],
    'official_model_id': os.environ['SLM_WM_TREE_RING_OFFICIAL_MODEL_ID'],
    'official_python_executable': os.environ['SLM_WM_TREE_RING_OFFICIAL_PYTHON_EXECUTABLE'] or 'current_python',
    'run_command': os.environ['SLM_WM_TREE_RING_OFFICIAL_RUN_COMMAND'],
    'summary_import_path': os.environ['SLM_WM_TREE_RING_OFFICIAL_SUMMARY_IMPORT_PATH'],
    'log_import_path': os.environ['SLM_WM_TREE_RING_OFFICIAL_LOG_IMPORT_PATH'],
})


In [ ]:
%pip install -q --upgrade packaging huggingface_hub


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
from pathlib import Path

source_dir = Path(os.environ['SLM_WM_TREE_RING_OFFICIAL_SOURCE_DIR'])
requirements_path = source_dir / 'requirements.txt'
entrypoint_path = source_dir / 'run_tree_ring_watermark.py'
print({
    'source_dir': str(source_dir),
    'source_dir_exists_before_helper': source_dir.exists(),
    'entrypoint_exists_before_helper': entrypoint_path.exists(),
    'requirements_exists_before_helper': requirements_path.exists(),
    'source_prepare_policy': 'helper_will_clone_from_external_baseline_source_registry_when_missing',
})
if requirements_path.exists():
    print(requirements_path.read_text(encoding='utf-8')[:4000])


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)


In [ ]:
from paper_workflow.colab_utils.tree_ring_official_reference import run_default_tree_ring_official_reference_plan

tree_ring_official_reference_summary = run_default_tree_ring_official_reference_plan(root='.')
tree_ring_official_reference_summary


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.tree_ring_official_reference import package_tree_ring_official_reference_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ.get('SLM_WM_TREE_RING_OFFICIAL_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/tree_ring_official_reference')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'tree_ring_official_reference_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_tree_ring_official_reference_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/tree_ring_official_reference').glob('*')):
    if result_path.is_file() and result_path.suffix.lower() in {'.json', '.jsonl', '.txt'}:
        print(result_path)
        print(result_path.read_text(encoding='utf-8', errors='ignore')[:4000])

assert tree_ring_official_reference_summary['sample_count'] == 5, tree_ring_official_reference_summary
assert tree_ring_official_reference_summary['run_decision'] == 'pass', tree_ring_official_reference_summary
assert tree_ring_official_reference_summary['reference_import_ready'] is True, tree_ring_official_reference_summary
assert tree_ring_official_reference_summary['governed_reference_record_count'] > 0, tree_ring_official_reference_summary
